In [ ]:

import json
import os
from pathlib import Path


def count_dataset_records(rde_path):
    """
    Go through all datasets in the rde folder and count the number of records.
    
    Args:
        rde_path: Path to the rde folder
        
    Returns:
        List of dictionaries with 'dataset_name' and 'record_count' keys
    """
    datasets_path = Path(rde_path) / "datasets"
    results = []
    
    # Check if the datasets folder exists
    if not datasets_path.exists():
        print(f"Error: {datasets_path} does not exist")
        return results
    
    # Iterate through all folders in the datasets directory
    for dataset_folder in sorted(datasets_path.iterdir()):
        # Skip files, only process directories
        if not dataset_folder.is_dir():
            continue
        
        # Skip hidden folders
        if dataset_folder.name.startswith('.'):
            continue
        
        # Path to the historical_records.json file
        historical_records_path = dataset_folder / "historical_records.json"
        
        # Check if the file exists
        if historical_records_path.exists():
            try:
                with open(historical_records_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                
                # Get the rde_objects array
                rde_objects = data.get('rde_objects', [])
                record_count = len(rde_objects)
                
                results.append({
                    'dataset_name': dataset_folder.name,
                    'record_count': record_count
                })
                
                print(f"{dataset_folder.name}: {record_count:,} records")
                
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON in {dataset_folder.name}: {e}")
            except Exception as e:
                print(f"Error processing {dataset_folder.name}: {e}")
        else:
            print(f"No historical_records.json found in {dataset_folder.name}")
    
    return results


if __name__ == "__main__":
    # Get the path to the rde folder (relative to this script)
    # script_dir = Path(__file__).parent
    rde_path =  "../rde"
    
    print("Counting records in datasets...\n")
    results = count_dataset_records(rde_path)
    
    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    
    if results:
        total_records = sum(r['record_count'] for r in results)
        print(f"\nTotal datasets: {len(results)}")
        print(f"Total records: {total_records:,}")
        
        # Print sorted by record count (descending)
        print("\nDatasets sorted by record count:")
        for item in sorted(results, key=lambda x: x['record_count'], reverse=True):
            print(f"  {item['dataset_name']}: {item['record_count']:,}")
    else:
        print("No datasets found or processed.")

In [ ]:
import pandas as pd
pd.DataFrame(results).to_csv("dataset_record_counts.csv", index=False)

In [ ]:
import json
import os
from pathlib import Path

with open('../rde/datasets/europeana-pipeline-postcards/src/wam_22k_results_internal.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

len(data)

In [ ]:
#!/usr/bin/env python3
"""
Extract coordinates from all RDE dataset observations.
This script processes all datasets in the rde/datasets directory,
reads their observations.json files, and extracts coordinates into a DataFrame.
"""

import json
import os
from pathlib import Path
import pandas as pd


def extract_coordinates_from_datasets(rde_base_path):
    """
    Extract coordinates from all dataset observations.
    
    Args:
        rde_base_path: Path to the rde directory
        
    Returns:
        pandas.DataFrame: DataFrame with columns: dataset_name, longitude, latitude
    """
    results = []
    datasets_path = Path(rde_base_path) / "datasets"
    
    # Iterate through all dataset directories
    for dataset_dir in datasets_path.iterdir():
        if not dataset_dir.is_dir() or dataset_dir.name.startswith('.'):
            continue
            
        dataset_name = dataset_dir.name
        observations_file = dataset_dir / "observations.json"
        
        # Check if observations.json exists
        if not observations_file.exists():
            print(f"⚠️  No observations.json found for {dataset_name}")
            continue
        
        # Read and parse the JSON file
        try:
            with open(observations_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract rde_objects array
            rde_objects = data.get('rde_objects', [])
            
            if not rde_objects:
                print(f"⚠️  No rde_objects found in {dataset_name}")
                continue
            
            # Process each observation object
            for obj in rde_objects:
                geometry = obj.get('geometry')
                
                if geometry and geometry.get('type') == 'Point':
                    coordinates = geometry.get('coordinates')
                    if coordinates and len(coordinates) >= 2:
                        longitude, latitude = coordinates[0], coordinates[1]
                        results.append((dataset_name, longitude, latitude))
                elif geometry and geometry.get('type') in ['Polygon', 'MultiPolygon', 'LineString']:
                    # For non-point geometries, we could extract centroids or skip
                    # For now, we'll skip them
                    pass
            
            print(f"✓ Processed {dataset_name}: {len([r for r in results if r[0] == dataset_name])} observations")
            
        except json.JSONDecodeError as e:
            print(f"❌ Error reading JSON for {dataset_name}: {e}")
        except Exception as e:
            print(f"❌ Error processing {dataset_name}: {e}")
    
    # Convert results to DataFrame
    df = pd.DataFrame(results, columns=['dataset_name', 'longitude', 'latitude'])
    
    return df


def main():
    """Main function to execute the script."""
    # Get the script directory and navigate to rde
    rde_path =  Path("../rde")
    
    if not rde_path.exists():
        print(f"❌ RDE directory not found at {rde_path}")
        return
    
    print(f"Extracting coordinates from datasets in: {rde_path}")
    print("=" * 80)
    
    # Extract coordinates
    df = extract_coordinates_from_datasets(rde_path)
    
    print("=" * 80)
    print(f"\n📊 Summary:")
    print(f"Total observations: {len(df)}")
    print(f"Datasets processed: {df['dataset_name'].nunique()}")
    print(f"\nObservations per dataset:")
    print(df['dataset_name'].value_counts().to_string())
    
    # Display first few rows
    print(f"\n📋 First 10 rows:")
    print(df.head(10).to_string(index=False))
    
    return df


if __name__ == "__main__":
    df = main()


In [ ]:
dfeuro = pd.read_csv('../step_3_geo_results.csv')


In [ ]:
dfeuro = dfeuro[dfeuro['city_loc'].notnull()].copy()

In [ ]:
dfeuro['longitude'], dfeuro['latitude'] = zip(*dfeuro['city_loc'].apply(lambda loc: (eval(loc)[1], eval(loc)[0]) if loc else (None, None)))

In [ ]:
dfeuro['dataset_name'] = 'europeana-pipeline-postcards'

In [ ]:
df_res = pd.concat([df, dfeuro[['dataset_name', 'longitude', 'latitude']]], ignore_index=True)

In [ ]:
remove_datasets = [
    'dresden-europeana-postcards',
    'paris-europeana-postcards',
    'venice-europeana-postcards',
    'venice-1808-street-network',
    'vaud-raimunds-hotel-journey'
]

df_res = df_res[~df_res['dataset_name'].isin(remove_datasets)]

In [ ]:
import folium
from folium.plugins import FeatureGroupSubGroup
import numpy as np

df_res = df_res[df_res['dataset_name'] == 'europeana-pipeline-postcards']
# Count points sharing the same coordinates within each dataset
df_grouped = df_res.groupby(['dataset_name', 'longitude', 'latitude']).size().reset_index(name='count')

# Get unique datasets and count total records per dataset
dataset_counts = df_res.groupby('dataset_name').size().sort_values(ascending=False)
datasets = dataset_counts.index.tolist()  # Sorted by record count (descending)

# Assign colors to datasets
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', 
          '#a65628', '#f781bf', '#999999', '#66c2a5', '#fc8d62', '#8da0cb', 
          '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3', '#1b9e77']

# Create a color mapping for datasets
dataset_colors = {dataset: colors[i % len(colors)] for i, dataset in enumerate(datasets)}

# Calculate center of the map (mean of all coordinates)
center_lat = df_grouped['latitude'].mean()
center_lon = df_grouped['longitude'].mean()

# Create base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles='CartoDB positron')

# Create a feature group for each dataset (for layer control)
# Process in order of most to least records (they'll be drawn in this order)
for dataset in datasets:
    # Filter data for this dataset
    dataset_data = df_grouped[df_grouped['dataset_name'] == dataset]
    
    # Get color for this dataset
    color = dataset_colors[dataset]
    
    # Create legend label with color indicator
    legend_label = f'<span style="color:{color};">⬤</span> {dataset} ({dataset_counts[dataset]:,})'
    
    # Create a feature group for this dataset
    fg = folium.FeatureGroup(name=legend_label, show=True)
    
    # Add circle markers
    for _, row in dataset_data.iterrows():
        # Scale radius based on count (sqrt to make it more visually balanced)
        radius = 1# np.sqrt(row['count']) * (1 if row['count'] < 16 else 0.25)  # scaled by sqrt with min radius
        
        # Create circle marker
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=radius,
            color=color,
            fill=True,
            fillColor=color,
            fillOpacity=0.6,
            weight=1,
            popup=f"<b>{dataset}</b><br>Count: {row['count']}<br>"
        ).add_to(fg)
    
    # Add the feature group to the map
    fg.add_to(m)

# Add layer control with checkboxes
# folium.LayerControl(collapsed=False).add_to(m)

# Display the map
# m

In [ ]:
import folium
import json
import numpy as np

# Use all datasets or filter as needed
df_res_viz = df_res  # or df_res[df_res['dataset_name'] == 'europeana-pipeline-postcards']

# Count points sharing the same coordinates within each dataset
df_grouped = df_res_viz.groupby(['dataset_name', 'longitude', 'latitude']).size().reset_index(name='count')

# Get unique datasets and count total records per dataset
dataset_counts = df_res_viz.groupby('dataset_name').size().sort_values(ascending=False)
datasets = dataset_counts.index.tolist()

# Assign colors to datasets
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', 
          '#a65628', '#f781bf', '#999999', '#66c2a5', '#fc8d62', '#8da0cb', 
          '#e78ac3', '#a6d854', '#ffd92f', '#e5c494', '#b3b3b3', '#1b9e77']

dataset_colors = {dataset: colors[i % len(colors)] for i, dataset in enumerate(datasets)}

# Calculate center of the map
center_lat = df_grouped['latitude'].mean()
center_lon = df_grouped['longitude'].mean()

# Create base map
m_static = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles='CartoDB positron')

# Process each dataset as a separate GeoJSON layer
for dataset in datasets:
    dataset_data = df_grouped[df_grouped['dataset_name'] == dataset]
    color = dataset_colors[dataset]
    
    # Create GeoJSON feature collection for this dataset
    features = []
    for _, row in dataset_data.iterrows():
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row['longitude'], row['latitude']]
            },
            "properties": {
                "style": {
                    "color": color,
                    "fillColor": color,
                    "fillOpacity": 0.6,
                    "weight": 1,
                    "radius": 1
                }
            }
        }
        features.append(feature)
    
    geojson_data = {
        "type": "FeatureCollection",
        "features": features
    }
    
    # Create legend label with color indicator
    legend_label = f'<span style="color:{color};">⬤</span> {dataset} ({dataset_counts[dataset]:,})'
    
    # Add GeoJSON layer - style properties are in the GeoJSON itself
    folium.GeoJson(
        geojson_data,
        name=legend_label,
        show=True,
        marker=folium.CircleMarker(
            radius=1,
            fill=True,
            fillOpacity=0.6,
            weight=1
        ),
        style_function=lambda feature: feature['properties']['style']
    ).add_to(m_static)

# Add layer control
folium.LayerControl(collapsed=False).add_to(m_static)

print(f"Total datasets: {len(datasets)}")
print(f"Total points: {len(df_grouped):,}")
m_static

In [ ]:
# Save the static GeoJSON-based map
m_static.save('map_static_geojson.html')

# Compare file size with the original approach
import os

print("File size comparison:")
print("-" * 60)
if os.path.exists('dataset_observations_map_11.html'):
    size_mb = os.path.getsize('dataset_observations_map_11.html') / (1024 * 1024)
    print(f"{'Original (CircleMarkers)':40s}: {size_mb:8.2f} MB")

if os.path.exists('map_static_geojson.html'):
    size_mb = os.path.getsize('map_static_geojson.html') / (1024 * 1024)
    print(f"{'Static GeoJSON':40s}: {size_mb:8.2f} MB")
print("-" * 60)

In [ ]:
m.save('dataset_observations_map_11.html')